In [3]:
#aux functions: move, cut, vote?

#okay so this is how a mixed move works moving forward, on an actual time stamp series
def tracemove(s, d, i, l): 
  if i < len(l): 
    l[i] = l[i] + s
    while i < len(l) : 
      l[i] = l[i] + d
      i = i+1

#this is how mixed moves operate when propogating backwards on a delay series
def delmove(s, d, i, l) : 
  if i > 0 : 
    l[i-1] = l[i-1] + s
  l[i] = l[i] - s + d

#aux constructs : line model, tree word, tree model, tpn 

#this is the linear causal process of a tpn without and branches or synchrony
#essentially an unrolled automaton run, remembering the guards checked along the way 
#describe unrolling process in short somewhere? 
class Linemodel : 
  def __init__(self, length):
    self.l = length
    self.g = [None]*self.l

  def guard(self, pos, eft, lft):
    self.g[pos] = (eft, lft)

  def run(self, word): 
    i = 0
    prev = 0
    for x in word : 
      if (x - prev) > self.g[i][1] or (x - prev) < self.g[i][0] :  
        return "reject"
        break 
      prev = x
      i = i+1
    return "accept"

      
      




In [ ]:
#mixed forward lines 

def mixforward(og, os) : 
  s = os.copy()
  g = og.copy()
  cost = 0 
  stamp = 0 
  delay = 0
  i = 0
  #print(i , cost, g, s )
  while i < len(s)-1: 
    curr = g[i] - s[i]
    next = g[i+1] - s[i+1]
    if curr*next < 0: 
      tracemove(curr, 0, i, s)
    elif abs(next) < abs(curr) : 
      tracemove(curr-next, next, i, s)
    else: 
      tracemove(0, curr, i, s)
    cost = cost + abs(curr)
    #print(i , cost, g, s )
    i = i+1
  cost = cost + abs(g[i] - s[i])
  return cost


In [ ]:
#mixed backward lines 
def mixbackward(og, os) : 
  g = og.copy()
  s = os.copy()

  g.insert(0, 0)
  s.insert(0,0)
  delg = []
  dels = []
  for i in range(len(g)-1): 
    delg.append(g[i+1] - g[i])
    dels.append(s[i+1] - s[i])
  cost = 0 
  stamp = 0 
  delay = 0
  i = len(dels)-1
  #print(i, cost, delg, dels)

  while i > 0 : 
    curr = delg[i] - dels[i]
    next = delg[i-1] - dels[i-1]
    if curr*next >= 0 : 
      delmove(0, curr, i, dels)
    elif abs(curr) < abs(next) : 
      delmove(-curr, 0, i, dels)
    else : 
      delmove(next, curr - next, i, dels)
    cost = cost + abs(curr)
    #print(i, cost, delg, dels)
    i = i-1
  cost = cost + abs(delg[i] - dels[i])
  return cost


In [ ]:
#@title
#testing whether backwards and forwards mixed algos agree
import numpy as np
from tqdm import tqdm

right = 0

for i in tqdm(range(100000)):
  observation = list(np.random.randint(100, size=(50,)))
  model = list(np.random.randint(10,  size=(50,)))
  forward_algo = mixforward(model, observation)
  backward_algo = mixbackward(model, observation)
  if forward_algo != backward_algo:
    print("Run Failed")
    print(model, observation)
    print(forward_algo, backward_algo)
    break
  else : 
    right = right + 1

print("     ")
print("Successful Runs:", right)


100%|██████████| 100000/100000 [01:01<00:00, 1635.00it/s]

     
Successful Runs: 100000


In [9]:
#stamp algo goes here

#an object to handle piecewise-linear convex graphs 
class PWLCGraph : 
  #start with the zero function on the appropriate domain 
  def __init__(self, domain): 
    self.left = domain[0]
    self.right = domain[1]
    self.segments = [[self.left, 0, self.right]]
    #nadir is the rightmost endpoint of a negative slope segment, where the next flat segment should start
    self.nadir = 0 
    #begin stores the height of the leftmost point of the curve
    self.begin = 0 

  def insertflatseg(self, length): 
    i = self.nadir  
    flag = 0
    if i == len(self.segments) : 
      low = self.right 
      self.segments.append([low, 0, low + length])
    else : 
      while i < len(self.segments): 
        if self.segments[i][1] == 0: 
          self.segments[i][2] = self.segments[i][2] + length 
          flag = 1 
        elif flag == 0 and self.segments[i][1] > 0 : 
          low = self.segments[i][0]
          slope = 0 
          high = low + length 
          self.segments.insert(i, [low, slope, high])
          flag = 1
        else : 
          self.segments[i][0] = self.segments[i][0] + length 
          self.segments[i][2] = self.segments[i][2] + length
        i = i+1
    self.right = self.right + length

  def min(self, interval) : 
    a = interval[0]
    b = interval[1]
    i = 0 
    for i in range(len(self.segments)): 
      self.segments[i][0] = self.segments[i][0] + a
      self.segments[i][2] = self.segments[i][2] + a
    self.left = self.left + a
    self.right = self.right + a
    if b != a : 
      self.insertflatseg(b-a)

  def addmod(self, pivot) : 
    a = self.left 
    b = self.right 
    i = 0 
    self.begin = self.begin + abs(a - pivot)
    while i < len(self.segments): 
      if self.segments[i][2] <= pivot : 
        self.segments[i][1] = self.segments[i][1] - 1
      elif self.segments[i][0] < pivot and self.segments[i][2] > pivot : 
        slope = self.segments[i][1]
        low = self.segments[i][0]
        high = self.segments[i][2]
        self.segments.insert(i+1, [pivot, slope, high])
        self.segments[i][2] = pivot
        self.segments[i][1] = slope - 1
      else : 
        self.segments[i][1] = self.segments[i][1] + 1
      i = i+1
    i = 0 
    while i < len(self.segments) and self.segments[i][1] < 0 : 
      i = i + 1
    self.nadir = i


#stamp align cost < also word : add backtrack within stampcost later, need slope/point arrays>
def stampcost(net, trace) : 
  graphlist = []
  a = net.g[0][0] 
  b = net.g[0][1]
  graph = PWLCGraph([a, b])
  graph.addmod(trace[0])
  graphlist.append(graph)
  for i in range(1, net.l) : 
    a = net.g[i][0]
    b = net.g[i][1]
    graph.min([a,b])
    graph.addmod(trace[i])
    graphlist.append(graph)
  s = graph.segments[0][1]
  k = 0 
  cost = graph.begin
  
  while k < graph.nadir: 
    s = graph.segments[k][1]
    cost = cost + (s*(graph.segments[k][2] - graph.segments[k][0]))
    k = k + 1
  return cost 



In [10]:
#@title
#test model 
import math

n = Linemodel(5)
n.guard(0, 3, 5)
n.guard(1, 2, 4)
n.guard(2, 0, 0)
n.guard(3, 0, 10)
n.guard(4, 0, 1)

obs = [4, 5, 6, 7, 9]

stampcost(n, obs)

2

In [ ]:
#mixed trees 

In [ ]:
#delay align


from collections import Counter
from copy import deepcopy


class Marking(Counter):
    pass

    def __hash__(self):
        r = 0
        for p in self.items():
            r += 31 * hash(p[0]) * p[1]
        return r

    def __eq__(self, other):
        if not self.keys() == other.keys():
            return False
        for p in self.keys():
            if other.get(p) != self.get(p):
                return False
        return True

    def __le__(self, other):
        if not self.keys() <= other.keys():
            return False
        for p in self.keys():
            if other.get(p) < self.get(p):
                return False
        return True

    def __add__(self, other):
        m = Marking()
        for p in self.items():
            m[p[0]] = p[1]
        for p in other.items():
            m[p[0]] += p[1]
        return m

    def __sub__(self, other):
        m = Marking()
        for p in self.items():
            m[p[0]] = p[1]
        for p in other.items():
            m[p[0]] -= p[1]
            if m[p[0]] == 0:
                del m[p[0]]
        return m

    def __repr__(self):
        # return str([str(p.name) + ":" + str(self.get(p)) for p in self.keys()])
        # The previous representation had a bug, it took into account the order of the places with tokens
        return str([str(p.name) + ":" + str(self.get(p)) for p in sorted(list(self.keys()), key=lambda x: x.name)])

    def __deepcopy__(self, memodict={}):
        marking = Marking()
        memodict[id(self)] = marking
        for place in self:
            place_occ = self[place]
            new_place = memodict[id(place)] if id(place) in memodict else PetriNet.Place(place.name,
                                                                                         properties=place.properties)
            marking[new_place] = place_occ
        return marking


class PetriNet(object):
    class Place(object):

        def __init__(self, name, in_arcs=None, out_arcs=None, properties=None):
            self.__name = name
            self.__in_arcs = set() if in_arcs is None else in_arcs
            self.__out_arcs = set() if out_arcs is None else out_arcs
            self.__properties = dict() if properties is None else properties

        def __set_name(self, name):
            self.__name = name

        def __get_name(self):
            return self.__name

        def __get_out_arcs(self):
            return self.__out_arcs

        def __get_in_arcs(self):
            return self.__in_arcs

        def __get_properties(self):
            return self.__properties

        def __repr__(self):
            return str(self.name)

        def __eq__(self, other):
            # keep the ID for now in places
            return id(self) == id(other)

        def __hash__(self):
            # keep the ID for now in places
            return id(self)

        def __deepcopy__(self, memodict={}):
            if id(self) in memodict:
                return memodict[id(self)]
            new_place = PetriNet.Place(self.name, properties=self.properties)
            memodict[id(self)] = new_place
            for arc in self.in_arcs:
                new_arc = deepcopy(arc, memo=memodict)
                new_place.in_arcs.add(new_arc)
            for arc in self.out_arcs:
                new_arc = deepcopy(arc, memo=memodict)
                new_place.out_arcs.add(new_arc)
            return new_place

        name = property(__get_name, __set_name)
        in_arcs = property(__get_in_arcs)
        out_arcs = property(__get_out_arcs)
        properties = property(__get_properties)

    class Transition(object):

        def __init__(self, name, label=None, in_arcs=None, out_arcs=None, properties=None):
            self.__name = name
            self.__label = None if label is None else label
            self.__in_arcs = set() if in_arcs is None else in_arcs
            self.__out_arcs = set() if out_arcs is None else out_arcs
            self.__properties = dict() if properties is None else properties

        def __set_name(self, name):
            self.__name = name

        def __get_name(self):
            return self.__name

        def __set_label(self, label):
            self.__label = label

        def __get_label(self):
            return self.__label

        def __get_out_arcs(self):
            return self.__out_arcs

        def __get_in_arcs(self):
            return self.__in_arcs

        def __get_properties(self):
            return self.__properties

        def __repr__(self):
            if self.label is None:
                return str(self.name)
            else:
                return str(self.label)

        def __eq__(self, other):
            # keep the ID for now in transitions
            return id(self) == id(other)

        def __hash__(self):
            # keep the ID for now in transitions
            return id(self)

        def __deepcopy__(self, memodict={}):
            if id(self) in memodict:
                return memodict[id(self)]
            new_trans = PetriNet.Transition(self.name, self.label, properties=self.properties)
            memodict[id(self)] = new_trans
            for arc in self.in_arcs:
                new_arc = deepcopy(arc, memo=memodict)
                new_trans.in_arcs.add(new_arc)
            for arc in self.out_arcs:
                new_arc = deepcopy(arc, memo=memodict)
                new_trans.out_arcs.add(new_arc)
            return new_trans

        name = property(__get_name, __set_name)
        label = property(__get_label, __set_label)
        in_arcs = property(__get_in_arcs)
        out_arcs = property(__get_out_arcs)
        properties = property(__get_properties)

    class Arc(object):

        def __init__(self, source, target, weight=1, properties=None):
            if type(source) is type(target):
                raise Exception('Petri nets are bipartite graphs!')
            self.__source = source
            self.__target = target
            self.__weight = weight
            self.__properties = dict() if properties is None else properties

        def __get_source(self):
            return self.__source

        def __get_target(self):
            return self.__target

        def __set_weight(self, weight):
            self.__weight = weight

        def __get_weight(self):
            return self.__weight

        def __get_properties(self):
            return self.__properties

        def __repr__(self):
            if type(self.source) is PetriNet.Transition:
                if self.source.label:
                    return "(t)" + str(self.source.label) + "->" + "(p)" + str(self.target.name)
                else:
                    return "(t)" + str(self.source.name) + "->" + "(p)" + str(self.target.name)
            if type(self.target) is PetriNet.Transition:
                if self.target.label:
                    return "(p)" + str(self.source.name) + "->" + "(t)" + str(self.target.label)
                else:
                    return "(p)" + str(self.source.name) + "->" + "(t)" + str(self.target.name)


        def __hash__(self):
            return id(self)

        def __eq__(self, other):
            return self.source == other.source and self.target == other.target

        def __deepcopy__(self, memodict={}):
            if id(self) in memodict:
                return memodict[id(self)]
            new_source = memodict[id(self.source)] if id(self.source) in memodict else deepcopy(self.source,
                                                                                                memo=memodict)
            new_target = memodict[id(self.target)] if id(self.target) in memodict else deepcopy(self.target,
                                                                                                memo=memodict)
            memodict[id(self.source)] = new_source
            memodict[id(self.target)] = new_target
            new_arc = PetriNet.Arc(new_source, new_target, weight=self.weight, properties=self.properties)
            memodict[id(self)] = new_arc
            return new_arc

        source = property(__get_source)
        target = property(__get_target)
        weight = property(__get_weight, __set_weight)
        properties = property(__get_properties)

    def __init__(self, name=None, places=None, transitions=None, arcs=None, properties=None):
        self.__name = "" if name is None else name
        self.__places = set() if places is None else places
        self.__transitions = set() if transitions is None else transitions
        self.__arcs = set() if arcs is None else arcs
        self.__properties = dict() if properties is None else properties

    def __get_name(self):
        return self.__name

    def __set_name(self, name):
        self.__name = name

    def __get_places(self):
        return self.__places

    def __get_transitions(self):
        return self.__transitions

    def __get_arcs(self):
        return self.__arcs

    def __get_properties(self):
        return self.__properties

    def __hash__(self):
        ret = 0
        for p in self.places:
            ret += hash(p)
            ret = ret % 479001599
        for t in self.transitions:
            ret += hash(t)
            ret = ret % 479001599
        return ret

    def __eq__(self, other):
        # for the Petri net equality keep the ID for now
        return id(self) == id(other)

    def __deepcopy__(self, memodict={}):
        from pm4py.objects.petri_net.utils.petri_utils import add_arc_from_to
        this_copy = PetriNet(self.name)
        memodict[id(self)] = this_copy
        for place in self.places:
            place_copy = PetriNet.Place(place.name, properties=place.properties)
            this_copy.places.add(place_copy)
            memodict[id(place)] = place_copy
        for trans in self.transitions:
            trans_copy = PetriNet.Transition(trans.name, trans.label, properties=trans.properties)
            this_copy.transitions.add(trans_copy)
            memodict[id(trans)] = trans_copy
        for arc in self.arcs:
            add_arc_from_to(memodict[id(arc.source)], memodict[id(arc.target)], this_copy, weight=arc.weight)
        return this_copy

    name = property(__get_name, __set_name)
    places = property(__get_places)
    transitions = property(__get_transitions)
    arcs = property(__get_arcs)
    properties = property(__get_properties)